# FIFA World Cup 2026 — Data Analysis & Match Predictor

In [16]:
import pandas as pd
import requests
import os
from dotenv import load_dotenv

## 1. Load Results

In [17]:
df = pd.read_csv('data/results.csv')
print("Shape of the DataFrame:", df.shape)
df.head()

Shape of the DataFrame: (49378, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


## 2. Basic OPerations

In [18]:
print("Columns in the DataFrame:", df.columns)
print("Data types of each column:\n", df.dtypes)
print("Summary statistics of the DataFrame:\n", df.describe())
print("Number of unique values in each column:\n", df.nunique())
print("Number of missing values in each column:\n", df.isnull().sum())


Columns in the DataFrame: Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral'],
      dtype='str')
Data types of each column:
 date              str
home_team         str
away_team         str
home_score    float64
away_score    float64
tournament        str
city              str
country           str
neutral          bool
dtype: object
Summary statistics of the DataFrame:
          home_score    away_score
count  49306.000000  49306.000000
mean       1.757393      1.182209
std        1.774934      1.402759
min        0.000000      0.000000
25%        1.000000      0.000000
50%        1.000000      1.000000
75%        2.000000      2.000000
max       31.000000     21.000000
Number of unique values in each column:
 date          16463
home_team       327
away_team       321
home_score       26
away_score       22
tournament      199
city           2139
country         269
neutral           2
dtype: int64
Number of missin

In [19]:
print("Top 30 tournaments in the dataset:")
print(df['tournament'].value_counts().head(30))

Top 30 tournaments in the dataset:
tournament
Friendly                                18301
FIFA World Cup qualification             8771
UEFA Euro qualification                  2824
African Cup of Nations qualification     2327
FIFA World Cup                           1036
Copa América                              869
African Cup of Nations                    845
AFC Asian Cup qualification               829
UEFA Nations League                       658
CECAFA Cup                                620
CFU Caribbean Cup qualification           606
Merdeka Tournament                        599
British Home Championship                 523
CONCACAF Nations League                   422
AFC Asian Cup                             421
Gold Cup                                  420
Gulf Cup                                  410
Island Games                              394
UEFA Euro                                 388
Asian Games                               368
COSAFA Cup                        

In [20]:
df['date'] = pd.to_datetime(df['date'])
print("Oldest match date:", df['date'].min())
print("Most recent match date:", df['date'].max())
print("Unique teams in the dataset:", df['home_team'].nunique() + df['away_team'].nunique())

Oldest match date: 1872-11-30 00:00:00
Most recent match date: 2026-06-27 00:00:00
Unique teams in the dataset: 648


##  3. Get only the competitive matches since 2018

In [21]:
competitive_tournaments = [
    'FIFA World Cup',
    'FIFA World Cup qualification',
    'UEFA Euro qualification',
    'UEFA Euro',
    'UEFA Nations League',
    'African Cup of Nations qualification',
    'African Cup of Nations',
    'Copa América',
    'AFC Asian Cup qualification',
    'AFC Asian Cup',
    'CONCACAF Nations League',
    'Gold Cup',
    'Arab Cup',
    'Gulf Cup'
]
df_2018 = df[df['date'] >= '2018-01-01']
df_competitive = df_2018[df_2018['tournament'].isin(competitive_tournaments)]

print(f"Total matches since 2018: {len(df_2018)}")
print(f"After removing friendlies/minor tournaments: {len(df_competitive)}")
print(f"\nBreakdown:")
print(df_competitive['tournament'].value_counts())

Total matches since 2018: 8081
After removing friendlies/minor tournaments: 4959

Breakdown:
tournament
FIFA World Cup qualification            1767
UEFA Nations League                      658
African Cup of Nations qualification     560
UEFA Euro qualification                  501
CONCACAF Nations League                  422
African Cup of Nations                   208
FIFA World Cup                           200
Gold Cup                                 124
AFC Asian Cup qualification              118
AFC Asian Cup                            102
UEFA Euro                                102
Copa América                              86
Arab Cup                                  63
Gulf Cup                                  48
Name: count, dtype: int64


## 4. Check if there is enough data for WC 2026 Teams

In [22]:
all_teams = pd.concat([df_competitive['home_team'], df_competitive['away_team']]).unique()

print("\nAll teams in competitive dataset:")
print(sorted(all_teams))


All teams in competitive dataset:
['Afghanistan', 'Albania', 'Algeria', 'American Samoa', 'Andorra', 'Angola', 'Anguilla', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Aruba', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin', 'Bermuda', 'Bhutan', 'Bolivia', 'Bonaire', 'Bosnia and Herzegovina', 'Botswana', 'Brazil', 'British Virgin Islands', 'Brunei', 'Bulgaria', 'Burkina Faso', 'Burundi', 'Cambodia', 'Cameroon', 'Canada', 'Cape Verde', 'Cayman Islands', 'Central African Republic', 'Chad', 'Chile', 'China PR', 'Colombia', 'Comoros', 'Congo', 'Cook Islands', 'Costa Rica', 'Croatia', 'Cuba', 'Curaçao', 'Cyprus', 'Czech Republic', 'DR Congo', 'Denmark', 'Djibouti', 'Dominica', 'Dominican Republic', 'Ecuador', 'Egypt', 'El Salvador', 'England', 'Equatorial Guinea', 'Eritrea', 'Estonia', 'Eswatini', 'Ethiopia', 'Faroe Islands', 'Fiji', 'Finland', 'France', 'French Guiana', 'Gabon', 'Gambia', 'Georgia', 'Germany'

In [23]:
wc_2026_teams = [
    'Argentina', 'Australia', 'Austria', 'Algeria',
    'Belgium', 'Bosnia and Herzegovina', 'Brazil',
    'Canada', 'Cape Verde', 'Colombia', 'DR Congo', 'Croatia',
    'Curaçao', 'Czech Republic',
    'Egypt', 'Ecuador', 'England',
    'France',
    'Germany', 'Ghana',
    'Haiti',
    'Iran', 'Iraq', 'Ivory Coast',
    'Japan', 'Jordan',
    'Mexico', 'Morocco',
    'Netherlands', 'New Zealand', 'Norway',
    'Panama', 'Paraguay', 'Portugal',
    'Qatar',
    'Saudi Arabia', 'Scotland', 'Senegal',
    'South Africa', 'South Korea', 'Spain',
    'Sweden', 'Switzerland',
    'Tunisia', 'Turkey',
    'Uruguay', 'United States', 'Uzbekistan'
]

print(f"{'Team':<20} {'Matches':>10}")
print("-" * 32)
for team in wc_2026_teams:
    matches = df_competitive[
        (df_competitive['home_team'] == team) |
        (df_competitive['away_team'] == team)
    ]
    flag = " LOW" if len(matches) < 25 else ""
    print(f"{team:<20} {len(matches):>10}{flag}")

Team                    Matches
--------------------------------
Argentina                    68
Australia                    56
Austria                      72
Algeria                      72
Belgium                      82
Bosnia and Herzegovina         65
Brazil                       65
Canada                       65
Cape Verde                   51
Colombia                     60
DR Congo                     62
Croatia                      84
Curaçao                      44
Czech Republic               68
Egypt                        78
Ecuador                      54
England                      87
France                       87
Germany                      68
Ghana                        55
Haiti                        52
Iran                         55
Iraq                         70
Ivory Coast                  63
Japan                        59
Jordan                       52
Mexico                       71
Morocco                      80
Netherlands                  80
New Z

## 5. Save dataset with competitive matches since 2018

In [24]:
df_competitive.to_csv('data/competitive_matches.csv', index=False)
print(df_competitive.shape)
df_competitive.head()


(4959, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
41297,2018-01-02,Iraq,United Arab Emirates,0.0,0.0,Gulf Cup,Kuwait City,Kuwait,True
41298,2018-01-02,Oman,Bahrain,1.0,0.0,Gulf Cup,Kuwait City,Kuwait,True
41299,2018-01-05,Oman,United Arab Emirates,0.0,0.0,Gulf Cup,Kuwait City,Kuwait,True
41332,2018-03-22,Kyrgyzstan,Myanmar,5.0,1.0,AFC Asian Cup qualification,Incheon,South Korea,True
41403,2018-03-27,Afghanistan,Cambodia,2.0,1.0,AFC Asian Cup qualification,Dushanbe,Tajikistan,True


## 6. Fetch WC 2026 Fixtures 

In [25]:
load_dotenv()
API_KEY = os.getenv("API_KEY")
headers = {"X-Auth-Token": API_KEY}

r = requests.get(
    "https://api.football-data.org/v4/competitions/WC/matches",
    headers=headers,
    timeout=10
)

print(f"Status code: {r.status_code}")
data = r.json()
print(f"Keys in response: {list(data.keys())}")

Status code: 200
Keys in response: ['filters', 'resultSet', 'competition', 'matches']


In [26]:
fixtures = pd.DataFrame(data['matches'])
fixtures.head()

,area,competition,season,id,utcDate,status,matchday,stage,group,lastUpdated,homeTeam,awayTeam,score,odds,referees
0,"{'id': 2267, 'name': 'World', 'code': 'INT', '...","{'id': 2000, 'name': 'FIFA World Cup', 'code':...","{'id': 2398, 'startDate': '2026-06-11', 'endDa...",537327,2026-06-11T19:00:00Z,TIMED,1.0,GROUP_STAGE,GROUP_A,2026-06-04T00:20:23Z,"{'id': 769, 'name': 'Mexico', 'shortName': 'Me...","{'id': 774, 'name': 'South Africa', 'shortName...","{'winner': None, 'duration': 'REGULAR', 'fullT...",{'msg': 'Activate Odds-Package in User-Panel t...,[]
1,"{'id': 2267, 'name': 'World', 'code': 'INT', '...","{'id': 2000, 'name': 'FIFA World Cup', 'code':...","{'id': 2398, 'startDate': '2026-06-11', 'endDa...",537328,2026-06-12T02:00:00Z,TIMED,1.0,GROUP_STAGE,GROUP_A,2026-06-04T00:20:23Z,"{'id': 772, 'name': 'South Korea', 'shortName'...","{'id': 798, 'name': 'Czechia', 'shortName': 'C...","{'winner': None, 'duration': 'REGULAR', 'fullT...",{'msg': 'Activate Odds-Package in User-Panel t...,[]
2,"{'id': 2267, 'name': 'World', 'code': 'INT', '...","{'id': 2000, 'name': 'FIFA World Cup', 'code':...","{'id': 2398, 'startDate': '2026-06-11', 'endDa...",537333,2026-06-12T19:00:00Z,TIMED,1.0,GROUP_STAGE,GROUP_B,2026-06-04T00:20:23Z,"{'id': 828, 'name': 'Canada', 'shortName': 'Ca...","{'id': 1060, 'name': 'Bosnia-Herzegovina', 'sh...","{'winner': None, 'duration': 'REGULAR', 'fullT...",{'msg': 'Activate Odds-Package in User-Panel t...,[]
3,"{'id': 2267, 'name': 'World', 'code': 'INT', '...","{'id': 2000, 'name': 'FIFA World Cup', 'code':...","{'id': 2398, 'startDate': '2026-06-11', 'endDa...",537345,2026-06-13T01:00:00Z,TIMED,1.0,GROUP_STAGE,GROUP_D,2026-06-04T00:20:23Z,"{'id': 771, 'name': 'United States', 'shortNam...","{'id': 761, 'name': 'Paraguay', 'shortName': '...","{'winner': None, 'duration': 'REGULAR', 'fullT...",{'msg': 'Activate Odds-Package in User-Panel t...,[]
4,"{'id': 2267, 'name': 'World', 'code': 'INT', '...","{'id': 2000, 'name': 'FIFA World Cup', 'code':...","{'id': 2398, 'startDate': '2026-06-11', 'endDa...",537334,2026-06-13T19:00:00Z,TIMED,1.0,GROUP_STAGE,GROUP_B,2026-06-04T00:20:23Z,"{'id': 8030, 'name': 'Qatar', 'shortName': 'Qa...","{'id': 788, 'name': 'Switzerland', 'shortName'...","{'winner': None, 'duration': 'REGULAR', 'fullT...",{'msg': 'Activate Odds-Package in User-Panel t...,[]


In [27]:
fixtures[['utcDate', 'homeTeam', 'awayTeam', 'status']].head(10)

,utcDate,homeTeam,awayTeam,status
0,2026-06-11T19:00:00Z,"{'id': 769, 'name': 'Mexico', 'shortName': 'Me...","{'id': 774, 'name': 'South Africa', 'shortName...",TIMED
1,2026-06-12T02:00:00Z,"{'id': 772, 'name': 'South Korea', 'shortName'...","{'id': 798, 'name': 'Czechia', 'shortName': 'C...",TIMED
2,2026-06-12T19:00:00Z,"{'id': 828, 'name': 'Canada', 'shortName': 'Ca...","{'id': 1060, 'name': 'Bosnia-Herzegovina', 'sh...",TIMED
3,2026-06-13T01:00:00Z,"{'id': 771, 'name': 'United States', 'shortNam...","{'id': 761, 'name': 'Paraguay', 'shortName': '...",TIMED
4,2026-06-13T19:00:00Z,"{'id': 8030, 'name': 'Qatar', 'shortName': 'Qa...","{'id': 788, 'name': 'Switzerland', 'shortName'...",TIMED
5,2026-06-13T22:00:00Z,"{'id': 764, 'name': 'Brazil', 'shortName': 'Br...","{'id': 815, 'name': 'Morocco', 'shortName': 'M...",TIMED
6,2026-06-14T01:00:00Z,"{'id': 836, 'name': 'Haiti', 'shortName': 'Hai...","{'id': 8873, 'name': 'Scotland', 'shortName': ...",TIMED
7,2026-06-14T04:00:00Z,"{'id': 779, 'name': 'Australia', 'shortName': ...","{'id': 803, 'name': 'Turkey', 'shortName': 'Tu...",TIMED
8,2026-06-14T17:00:00Z,"{'id': 759, 'name': 'Germany', 'shortName': 'G...","{'id': 9460, 'name': 'Curaçao', 'shortName': '...",TIMED
9,2026-06-14T20:00:00Z,"{'id': 8601, 'name': 'Netherlands', 'shortName...","{'id': 766, 'name': 'Japan', 'shortName': 'Jap...",TIMED


In [29]:
import ast

fixtures = pd.read_csv('data/wc_2026_fixtures.csv')

fixtures['home_name'] = fixtures['homeTeam'].apply(
    lambda x: ast.literal_eval(x)['name'] if isinstance(x, str) else x['name']
)
fixtures['away_name'] = fixtures['awayTeam'].apply(
    lambda x: ast.literal_eval(x)['name'] if isinstance(x, str) else x['name']
)

print("Parsed. Columns now:", fixtures.columns.tolist())
print(fixtures['home_name'].head(5))

Parsed. Columns now: ['area', 'competition', 'season', 'id', 'utcDate', 'status', 'matchday', 'stage', 'group', 'lastUpdated', 'homeTeam', 'awayTeam', 'score', 'odds', 'referees', 'home_name', 'away_name']
0           Mexico
1      South Korea
2           Canada
3    United States
4            Qatar
Name: home_name, dtype: str


In [30]:
api_to_dataset = {
    'Bosnia-Herzegovina':  'Bosnia and Herzegovina',
    'Cape Verde Islands':  'Cape Verde',
    'Congo DR':            'DR Congo',
    'Czechia':             'Czech Republic',
}

fixtures['home_name'] = fixtures['home_name'].replace(api_to_dataset)
fixtures['away_name'] = fixtures['away_name'].replace(api_to_dataset)

# Save clean fixtures
fixtures.to_csv('data/wc_2026_fixtures.csv', index=False)
print(f"Fixtures saved: {len(fixtures)} matches")
# Verify — these 4 should not appear anymore
check = pd.concat([fixtures['home_name'], fixtures['away_name']]).dropna().unique()
print(sorted(check))

print(f"\nMatches by stage:")
print(fixtures['stage'].value_counts())

Fixtures saved: 104 matches
['Algeria', 'Argentina', 'Australia', 'Austria', 'Belgium', 'Bosnia and Herzegovina', 'Brazil', 'Canada', 'Cape Verde', 'Colombia', 'Croatia', 'Curaçao', 'Czech Republic', 'DR Congo', 'Ecuador', 'Egypt', 'England', 'France', 'Germany', 'Ghana', 'Haiti', 'Iran', 'Iraq', 'Ivory Coast', 'Japan', 'Jordan', 'Mexico', 'Morocco', 'Netherlands', 'New Zealand', 'Norway', 'Panama', 'Paraguay', 'Portugal', 'Qatar', 'Saudi Arabia', 'Scotland', 'Senegal', 'South Africa', 'South Korea', 'Spain', 'Sweden', 'Switzerland', 'Tunisia', 'Turkey', 'United States', 'Uruguay', 'Uzbekistan']

Matches by stage:
stage
GROUP_STAGE       72
LAST_32           16
LAST_16            8
QUARTER_FINALS     4
SEMI_FINALS        2
THIRD_PLACE        1
FINAL              1
Name: count, dtype: int64


## Data Loading and initial cleaning DONE

- Loaded the matches from 1872 to 2026
- Made new dataset with competitive matches since 2018(Relevant for analysis)
- Fetched WC 2026 Fixtures


| Item | Result |
|------|--------|
| Total dataset size | 49378 matches |
| Competitive matches since 2018 | 4959 matches |
| Teams with low data (< 25 matches) | 1 team |
| WC 2026 fixtures fetched | 104 matches |


